In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_ibca_probe import ViViTBackbone, ViViTIBCALinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-07 20:25:27,428]::2730006395::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# ViViT-IBCA Block-wise Linear Probe
# ============================================================
# Frozen backbone; only independent linear probe heads are trained.

PROBE_SEED = 42
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 120
patience = 20

# Use the checkpoint produced by the corresponding trained ViViT-IBCA run.
BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(PROBE_SEED)

g = torch.Generator()
g.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker,
    generator=g
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker
)

# Exactly the same ViViT-IBCA backbone configuration as the supplied notebook.
backbone = ViViTBackbone(
    2, 14, 14,
    2, 2, 2,
    5,
    96,
    6,
    8,
    384,
    model=3
).to(device)
ckpt = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
backbone.load_state_dict(state_dict, strict=True)

model = ViViTIBCALinearProbe(
    backbone=backbone,
    dim=96,
    num_outputs=5
).to(device)

assert sum(p.numel() for p in model.backbone.parameters() if p.requires_grad) == 0
print("Backbone trainable parameters: 0")
print("Probe trainable parameters:",
      sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad))

criterion = nn.L1Loss().to(device)
optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)

probe_names = list(model.probe_heads.keys())
best_mae = {name: float("inf") for name in probe_names}
best_epoch = {name: -1 for name in probe_names}
best_mean_val_mae = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):
    model.train()
    train_sum = {name: 0.0 for name in probe_names}

    for audiodata, fullshot, big_five_data in tqdm(
        train_dataloader, desc=f"Probe Train {epoch+1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        audiodata = audiodata.to(device)

        optimizer.zero_grad()
        outputs = model(fullshot, audiodata)
        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }
        sum(losses.values()).backward()
        optimizer.step()

        for name in probe_names:
            train_sum[name] += losses[name].item()

    train_mae = {
        name: train_sum[name] / len(train_dataloader)
        for name in probe_names
    }

    model.eval()
    val_abs_sum = {name: 0.0 for name in probe_names}
    val_count = 0

    with torch.no_grad():
        for audiodata, fullshot, big_five_data in tqdm(
            val_dataloader, desc=f"Probe Valid {epoch+1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            audiodata = audiodata.to(device)

            outputs = model(fullshot, audiodata)
            val_count += big_five_data.numel()

            for name in probe_names:
                val_abs_sum[name] += torch.abs(
                    outputs[name] - big_five_data
                ).sum().item()

    val_mae = {
        name: val_abs_sum[name] / val_count
        for name in probe_names
    }

    for name in probe_names:
        if val_mae[name] < best_mae[name]:
            best_mae[name] = val_mae[name]
            best_epoch[name] = epoch + 1

    mean_val_mae = float(np.mean([val_mae[n] for n in probe_names]))

    if mean_val_mae < best_mean_val_mae:
        best_mean_val_mae = mean_val_mae
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 80)
    print(
        f"Epoch {epoch+1:03d} | Mean Val MAE={mean_val_mae:.6f} | "
        f"Patience={no_improve_count}/{patience}"
    )
    for name in probe_names:
        print(
            f"{name:<8} | train MAE {train_mae[name]:.6f} | "
            f"val MAE {val_mae[name]:.6f} | "
            f"1-MAE {1-val_mae[name]:.6f} | "
            f"best 1-MAE {1-best_mae[name]:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch+1}. "
            f"Best mean validation MAE={best_mean_val_mae:.6f}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("FINAL ViViT-IBCA LINEAR PROBE RESULTS")
print("=" * 80)
print(f"{'Representation':<18}{'Best Epoch':>12}{'MAE':>12}{'1-MAE':>12}")
print("-" * 80)
for name in probe_names:
    print(
        f"{name:<18}{best_epoch[name]:>12d}"
        f"{best_mae[name]:>12.6f}{1-best_mae[name]:>12.6f}"
    )
print("=" * 80)

del model, backbone, optimizer, train_dataloader, val_dataloader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable parameters: 0
Probe trainable parameters: 2910


Probe Train 1/120: 100%|█████████▉| 1497/1498 [03:33<00:00,  5.09it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Probe Valid 1/120: 100%|██████████| 498/498 [01:38<00:00,  5.07it/s]



--------------------------------------------------------------------------------
Epoch 001 | Mean Val MAE=0.092073 | Patience=0/20
block1   | train MAE 0.077151 | val MAE 0.090141 | 1-MAE 0.909859 | best 1-MAE 0.909859
block2   | train MAE 0.075321 | val MAE 0.093879 | 1-MAE 0.906121 | best 1-MAE 0.906121
block3   | train MAE 0.078232 | val MAE 0.089808 | 1-MAE 0.910192 | best 1-MAE 0.910192
block4   | train MAE 0.072001 | val MAE 0.089914 | 1-MAE 0.910086 | best 1-MAE 0.910086
block5   | train MAE 0.069923 | val MAE 0.091217 | 1-MAE 0.908783 | best 1-MAE 0.908783
block6   | train MAE 0.072414 | val MAE 0.097477 | 1-MAE 0.902523 | best 1-MAE 0.902523


Probe Valid 2/120: 100%|██████████| 498/498 [01:41<00:00,  4.91it/s]



--------------------------------------------------------------------------------
Epoch 002 | Mean Val MAE=0.089687 | Patience=0/20
block1   | train MAE 0.067663 | val MAE 0.088973 | 1-MAE 0.911027 | best 1-MAE 0.911027
block2   | train MAE 0.066584 | val MAE 0.090473 | 1-MAE 0.909527 | best 1-MAE 0.909527
block3   | train MAE 0.067417 | val MAE 0.091179 | 1-MAE 0.908821 | best 1-MAE 0.910192
block4   | train MAE 0.067066 | val MAE 0.091228 | 1-MAE 0.908772 | best 1-MAE 0.910086
block5   | train MAE 0.065761 | val MAE 0.090028 | 1-MAE 0.909972 | best 1-MAE 0.909972
block6   | train MAE 0.065496 | val MAE 0.086242 | 1-MAE 0.913758 | best 1-MAE 0.913758


...


--------------------------------------------------------------------------------
Epoch 027 | Mean Val MAE=0.091714 | Patience=18/20
block1   | train MAE 0.063152 | val MAE 0.089844 | 1-MAE 0.910156 | best 1-MAE 0.914326
block2   | train MAE 0.062809 | val MAE 0.089219 | 1-MAE 0.910781 | best 1-MAE 0.913965
block3   | train MAE 0.063485 | val MAE 0.091203 | 1-MAE 0.908797 | best 1-MAE 0.916011
block4   | train MAE 0.063662 | val MAE 0.090692 | 1-MAE 0.909308 | best 1-MAE 0.916425
block5   | train MAE 0.064149 | val MAE 0.095662 | 1-MAE 0.904338 | best 1-MAE 0.915808
block6   | train MAE 0.063863 | val MAE 0.093665 | 1-MAE 0.906335 | best 1-MAE 0.916411


Probe Valid 28/120: 100%|██████████| 498/498 [01:15<00:00,  6.63it/s]



--------------------------------------------------------------------------------
Epoch 028 | Mean Val MAE=0.086837 | Patience=19/20
block1   | train MAE 0.063316 | val MAE 0.088187 | 1-MAE 0.911813 | best 1-MAE 0.914326
block2   | train MAE 0.062781 | val MAE 0.086003 | 1-MAE 0.913997 | best 1-MAE 0.913997
block3   | train MAE 0.063633 | val MAE 0.086167 | 1-MAE 0.913833 | best 1-MAE 0.916011
block4   | train MAE 0.063868 | val MAE 0.086624 | 1-MAE 0.913376 | best 1-MAE 0.916425
block5   | train MAE 0.063736 | val MAE 0.088932 | 1-MAE 0.911068 | best 1-MAE 0.915808
block6   | train MAE 0.063684 | val MAE 0.085111 | 1-MAE 0.914889 | best 1-MAE 0.916411


Probe Valid 29/120: 100%|██████████| 498/498 [01:15<00:00,  6.60it/s]



--------------------------------------------------------------------------------
Epoch 029 | Mean Val MAE=0.090722 | Patience=20/20
block1   | train MAE 0.063530 | val MAE 0.090356 | 1-MAE 0.909644 | best 1-MAE 0.914326
block2   | train MAE 0.063148 | val MAE 0.088709 | 1-MAE 0.911291 | best 1-MAE 0.913997
block3   | train MAE 0.063331 | val MAE 0.087803 | 1-MAE 0.912197 | best 1-MAE 0.916011
block4   | train MAE 0.063745 | val MAE 0.087754 | 1-MAE 0.912246 | best 1-MAE 0.916425
block5   | train MAE 0.064244 | val MAE 0.094231 | 1-MAE 0.905769 | best 1-MAE 0.915808
block6   | train MAE 0.064032 | val MAE 0.095479 | 1-MAE 0.904521 | best 1-MAE 0.916411

Early stopping at epoch 29. Best mean validation MAE=0.085055

FINAL ViViT-IBCA LINEAR PROBE RESULTS
Representation      Best Epoch         MAE       1-MAE
--------------------------------------------------------------------------------
block1                      24    0.085674    0.914326
block2                      28    0.086003    